# P3. 블록 만들기 — 원문을 "검색 가능한 조각"으로 나누기

## 1. 블록이란?
- **블록 = 원문의 문단 한 개를 번호 붙여 저장한 조각**입니다.
- 나중에 검색과 근거 인용은 이 조각 단위로 합니다.
- 블록 하나는 아래 항목을 가집니다.

| 항목 | 뜻 | 예시 |
|---|---|---|
| `chunk_id` | 블록 고유 번호 | `kdca_measles_2026:p0093:b01` |
| `document_id` | 어느 문서인지 | `kdca_measles_2026` |
| `text` | 문단 원문 | "교내 홍역(의심)환자 …" |
| `pdf_page` | PDF 파일 기준 쪽 | 93 |
| `printed_page` | 책에 인쇄된 쪽 (확인 전엔 비움) | None |
| `section` | 어느 장·절인지 | "04 기관별 감염병대응 담당자 역할" |

## 2. 이번 작업의 순서
1. 원문 문단 목록 보기 (코드)
2. **내가 쓸 문단 번호 고르기** (직접 결정)
3. 블록 만들기 (코드가 원문과 대조)
4. 파일로 저장하고 5건 무작위 확인

> 목표: 홍역 93쪽·24쪽, 감염병예방법 18~19쪽에서 **5~10개 블록**을 만든다.

In [1]:
# 0) 준비: 도구 불러오기
from preprocess.prevention_legal.pages import _load_pages, DOCS, excerpt_in_page
import json, pathlib, random

OUT = pathlib.Path("../../data/interim/prevention_legal/blocks.jsonl")   # 저장 위치 (Git에는 안 올라감)
OUT.parent.mkdir(parents=True, exist_ok=True)

def paragraphs(key, page):
    """쪽 하나를 빈 줄 기준 문단 목록으로 나눈다. 너무 짧은 줄(10자 미만)은 뺀다."""
    raw = _load_pages(key)[page]
    parts = raw.split("\n\n")
    return [p.strip() for p in parts if len(p.strip()) >= 10]

def show(key, page):
    """문단에 번호를 붙여서 보여 준다. 번호로 골라서 쓴다."""
    for i, p in enumerate(paragraphs(key, page)):
        line = p[:90].replace("\n", " / ")
        print(f"[{i:2d}] {line}")

print("준비 완료. 문서 키:", list(DOCS))

준비 완료. 문서 키: ['law', 'measles', 'hfmd', 'vaccine']


## 3. 홍역 지침 93쪽 — 문단 목록 보기
아래 셀을 실행하면 93쪽의 문단이 **번호와 함께** 나옵니다. 번호를 보고 골라 보세요.

In [2]:
show("measles", 93)

[ 0] 4. 기관별 감염병대응 담당자 역할
[ 1] ### **04 기관별 감염병대응 담당자 역할**
[ 2] ##### ･ ･ **가. 초 중 고등학교 내 환자 발생 시 학교[담당교사] 역할**
[ 3] **[홍역의 특징]**
[ 4] - **홍역을 의심할 수 있는 증상** : **고열과 함께피부 발진** 이 나타나고, 기침, 콧물, 결막염이 동반될
[ 5] **전염성** : 홍역은 전염력이 매우 강하며 항체가 없는 사람이 노출되었을 때 90% 이상이
[ 6] **감염경로** : 환자와 직접 접촉하거나 호흡기 분비물 등을 통해 감염됩니다.
[ 7] **전염기간** : 환자에서 피부 발진이 발생하기 4일전부터 발생 후 4일까지 홍역바이러스 전파가
[ 8] **잠복기** : 바이러스에 감염되어 증상이 나타나기까지의 기간은 7~21일(평균 10~12일)입니다.
[ 9] - ･ ･ / **예방접종** : 홍역은 MMR(홍역 유행성이하선염 풍진) 백신 2회 접종으로 효과적으로 예방이
[10] 1. 교내 **홍역(의심)환자** **[*]** **발생 시 반드시 의료기관에서 진료 받도록 안내하고, 관할 보건소에 신고**
[11] - 고열, 발진과 함께 기침, 콧물, 결막염 등의 증상이 동반하는 경우
[12] 2. 의료기관 진료 결과, **홍역(의심) 진단을 받은 경우에는 전염기간(발진이 나타난 후 4일까지)**
[13] **동안 학교 등교 및 학원 등원하지 않도록 안내**
[14] - 단, 실험실 검사 결과 환자 아님으로 확인 시 익일 등교 안내
[15] 3. 홍역(의심) 진단을 받지 않았으나 **유사증상자가 2주 이내 2명 이상 발생** 할 경우 **보건소로 신고**
[16] ･ ･ / 4. 교내 **MMR(홍역** **유행성이하선염** **풍진) 2차 접종 미완료자** 가 있는 경우 감염 예방을 위해
[17] **예방접종을 완료할 필요가 있음을 안내**
[18] 5. **추가 환자 발생 동향** 에 대해 **보건소와 정보**
[19] **홍** / **역**
[20

### ✍️ 직접 고르기 (TODO)
- 93쪽에서 **학교 담당교사가 무엇을 하는지**가 적힌 문단 번호를 고르세요.
- 힌트: 제목(`[담당교사] 역할`)과 `1.`로 시작하는 항목 근처를 보세요.
- 골라서 아래 `pick_93` 목록에 번호를 적으세요. (예: `[5, 6]`)

In [ ]:
pick_93 = [10,11,15]          # TODO: 번호를 적으세요. 예) [5, 6]

<details>
<summary>📦 정답 보기 (직접 고른 뒤에 펼치세요)</summary>

- 교내 홍역(의심)환자 발생 시 안내·신고 항목은 **"1."로 시작하는 문단**입니다. 그 문단 번호를 골랐다면 맞습니다.
- 3번 항목(유사증상자 2주 이내 2명 이상)도 같은 쪽에 있어서 함께 고르면 좋습니다.
- 제목 문단(`[담당교사] 역할`)은 블록으로 만들지 않아도 됩니다. 대신 `section` 칸에 적습니다.

</details>

## 4. 24쪽 · 법 18쪽 · 법 19쪽 — 같은 방식으로 고르기

In [ ]:
show("measles", 24)

In [ ]:
pick_24 = [5]          # 의료기관 신고 (24쪽 (의사)환자 진단 → 24시간 이내 보건소 신고)


In [ ]:
show("law", 18)

In [ ]:
pick_18 = [11, 22]     # 제11조 ①(의사→의료기관장 보고), ③(의료기관장→보건소장 24시간 신고)


In [ ]:
show("law", 19)

In [ ]:
pick_19 = [20, 25, 26] # 제12조 ①(조건 문장 20) + 관리인 주체 문장(25·26, 줄바꿈으로 나뉜 한 문장)


## 5. 블록 만들기 (코드가 원문과 대조합니다)

아래 함수는 **문단 번호 → 블록**으로 바꿉니다. 블록을 만들 때마다 원문에 그 문단이 있는지 확인합니다.
- 원문에 없으면 `ValueError`로 알려 줍니다. (그러면 번호를 다시 확인하세요.)

In [ ]:
def make_block(key, page, idx, block_no, section=""):
    """문단 번호 하나 또는 여러 개(목록)를 블록 한 개로 바꾼다."""
    # 줄바꿈 때문에 한 문장이 여러 문단으로 나뉜 경우, 번호를 목록으로 적으면 한 블록으로 합쳐진다.
    doc_id, _ = DOCS[key]
    ps = paragraphs(key, page)
    idxs = idx if isinstance(idx, list) else [idx]
    text = " ".join(ps[i] for i in idxs)
    if not excerpt_in_page(text, key, page):      # 원문 대조
        raise ValueError(f"{key} {page}쪽 {idxs} 이 원문에서 확인되지 않습니다.")
    return {
        "chunk_id": f"{doc_id}:p{page:04d}:b{block_no:02d}",
        "document_id": doc_id,
        "text": text,
        "pdf_page": page,
        "printed_page": None,
        "section": section,
    }

### ✍️ 블록 목록 완성 (TODO)
아래 `plan` 목록에 `(문서 키, 쪽, 문단 번호, 블록 번호, 장·절 이름)` 튜플을 적으세요.
- 블록 번호는 **같은 쪽 안에서 위에서 아래 순서**로 `1, 2, 3 …`을 매깁니다.
- 예시 모양: `("measles", 93, 5, 1, "04 기관별 감염병대응 담당자 역할")`

In [ ]:
plan = [
    # ("measles", 93, 5, 1, "04 기관별 감염병대응 담당자 역할"),   # TODO: 예시를 참고해서 채우세요
]

In [ ]:
blocks = []
for key, page, idx, block_no, section in plan:
    blocks.append(make_block(key, page, idx, block_no, section))
print("만든 블록:", len(blocks), "개")
for b in blocks:
    print(b["chunk_id"], "|", b["text"][:50].replace("\n", " "))

## 6. 저장하고 확인하기
- 저장 파일: `data/interim/prevention_legal/blocks.jsonl` (한 줄에 블록 하나, Git에는 올라가지 않습니다.)
- 아래 셀은 **번호(`chunk_id`)가 중복되지 않는지** 확인하고 저장합니다.

In [ ]:
ids = [b["chunk_id"] for b in blocks]
assert len(ids) == len(set(ids)), "chunk_id 가 중복됩니다. 블록 번호를 확인하세요."

with OUT.open("w", encoding="utf-8") as f:
    for b in blocks:
        f.write(json.dumps(b, ensure_ascii=False) + "\n")
print("저장 완료:", OUT, "| 블록", len(blocks), "개")

### ✅ 무작위 5건 원문 대조 (지시서 §5 기준)
- 아래 셀이 블록 중 무작위 5개를 골라 **원문 쪽에서 다시 찾습니다.**
- 모두 "원문 확인"이 나와야 합니다. 하나라도 "확인 필요"면 그 블록을 다시 보세요.

In [ ]:
sample = random.sample(blocks, min(5, len(blocks)))
for b in sample:
    key = [k for k, (d, _) in DOCS.items() if d == b["document_id"]][0]
    ok = excerpt_in_page(b["text"], key, b["pdf_page"])
    print(("원문 확인   " if ok else "확인 필요   "), b["chunk_id"])

## 7. 완료 체크
- [ ] 홍역 93쪽에서 블록을 골랐다
- [ ] 홍역 24쪽, 법 18·19쪽에서 블록을 골랐다 (합계 5~10개)
- [ ] 블록 목록을 `plan`에 적고 `make_block`으로 만들었다
- [ ] 저장했고 `chunk_id`가 중복되지 않는다
- [ ] 무작위 5건이 모두 "원문 확인"이다

다음 단계: **P7 추출 프롬프트 v1** — 저장한 블록을 넣어서 학교 행동을 뽑는 프롬프트를 만듭니다.